# 05 — DALL-E 2 / unCLIP: Hierarchical Text-Conditional Image Generation

**Paper:** *Hierarchical Text-Conditional Image Generation with CLIP Latents* (Ramesh et al., 2022)  
**arXiv:** https://arxiv.org/abs/2204.06125  
**Also known as:** DALL-E 2 (OpenAI, April 2022)

---

## Motivation

DALL-E 1 used a dVAE + Transformer autoregressive decoder.  
DALL-E 2 asks: what if we could leverage **CLIP's rich image-text embedding space** for generation?

Key insight: CLIP maps images and text to the **same embedding space**.  
If we can:
1. Map text → CLIP image embedding (a **prior**)
2. Decode CLIP image embedding → pixel image (an **unCLIP** decoder)

Then we have text → image generation that inherits CLIP's semantic richness.

## Architecture: Two-Stage Pipeline

![unCLIP Pipeline](./figures/unclip-figurehead.png)

```
Text y  →  CLIP Text Encoder  →  z_t  (text embedding)
                                    │
                                    ▼
                              Prior P(z_i | z_t)
                              (Diffusion Transformer)
                                    │
                                    ▼  z_i  (CLIP image embedding)
                              Decoder D(x | z_i, y)
                              (Cascaded diffusion: 64²→256²→1024²)
                                    │
                                    ▼
                              Final Image (1024×1024)
```

### Stage 1 — Prior: Text → CLIP Image Embedding

Goal: generate CLIP image embedding `z_i` from text `y`.  
The prior is a **Diffusion Transformer** (or autoregressive model) trained on `(text, CLIP_image_embed)` pairs.

### Stage 2 — Decoder (unCLIP): CLIP Embedding → Image

A **cascaded diffusion model** conditioned on `z_i`:
- Step 1: 64×64 image
- Step 2: 256×256 upsampler
- Step 3: 1024×1024 upsampler

## Why "unCLIP"?

CLIP *encodes* images into embeddings.  
The decoder *inverts* this — mapping embeddings back to images — hence **un**CLIP.

Because CLIP embeddings are **not invertible** (many images share similar embeddings), the decoder uses diffusion to sample diverse images that all correspond to the same embedding.

This gives DALL-E 2 a powerful capability: **image variations** — given any image, encode it with CLIP, then sample new images from the same embedding.

## CLIP Contrastive Embedding Space

Before diving into code, recall how CLIP works:

```
Image I → Vision Encoder → z_i  ∈ R^d     (L2-normalized)
Text  T → Text  Encoder  → z_t  ∈ R^d     (L2-normalized)

CLIP loss (InfoNCE):
  maximize cosine(z_i, z_t)  for matching pairs
  minimize cosine(z_i, z_t)  for non-matching pairs
```

The prior must map `z_t → z_i` while preserving the geometric structure learned by CLIP.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

# ── Simplified CLIP-space Prior (Diffusion Transformer) ──

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
    t = t.float().view(-1, 1)
    args = t * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class PriorTransformerBlock(nn.Module):
    """
    Transformer block for the Prior model.
    Attends to both noisy z_i tokens AND text conditioning tokens.
    """
    def __init__(self, d_model=512, n_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn   = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm3 = nn.LayerNorm(d_model)
        mlp_dim = int(d_model * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, mlp_dim), nn.GELU(),
            nn.Linear(mlp_dim, d_model)
        )

    def forward(self, x, context):
        # Self-attention
        h = self.norm1(x)
        sa, _ = self.self_attn(h, h, h)
        x = x + sa
        # Cross-attention to text tokens
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, context, context)
        x = x + ca
        # FFN
        x = x + self.ffn(self.norm3(x))
        return x


class CLIPPrior(nn.Module):
    """
    Diffusion Prior: maps (noisy z_i, text z_t, timestep) → denoised z_i.
    
    Sequence = [timestep_token, noisy_z_i_token, CLIP_text_embed_token]
    This is similar to the causal transformer in the original DALL-E 2 prior.
    """
    def __init__(self, clip_dim=512, d_model=512, depth=6, n_heads=8, text_seq_len=77):
        super().__init__()
        self.clip_dim  = clip_dim
        self.proj_in   = nn.Linear(clip_dim, d_model)
        self.time_proj = nn.Sequential(nn.Linear(d_model, d_model), nn.SiLU(),
                                       nn.Linear(d_model, d_model))
        self.blocks    = nn.ModuleList([
            PriorTransformerBlock(d_model, n_heads) for _ in range(depth)
        ])
        self.norm_out  = nn.LayerNorm(d_model)
        self.proj_out  = nn.Linear(d_model, clip_dim)

    def forward(self, z_noisy, t, text_tokens):
        # z_noisy:     (B, clip_dim)  — noisy CLIP image embedding
        # t:           (B,)           — timestep
        # text_tokens: (B, L, d_model) — CLIP text encoder output (already projected)
        B = z_noisy.shape[0]
        
        z = self.proj_in(z_noisy).unsqueeze(1)       # (B, 1, d_model)
        t_emb = self.time_proj(sinusoidal_embedding(t, z.shape[-1])).unsqueeze(1)  # (B, 1, d)
        
        # Concatenate timestep + z tokens
        x = torch.cat([t_emb, z], dim=1)             # (B, 2, d_model)
        
        for block in self.blocks:
            x = block(x, text_tokens)
        
        x = self.norm_out(x)
        z_pred = self.proj_out(x[:, 1])              # take z position, project back
        return z_pred


# Test
prior = CLIPPrior(clip_dim=512, d_model=256, depth=4, n_heads=4)
z_noise = torch.randn(2, 512)    # (B, clip_dim)
t       = torch.randint(0, 1000, (2,))
ctx     = torch.randn(2, 77, 256) # (B, text_seq, d_model)
z_pred  = prior(z_noise, t, ctx)
print(f"Prior input z:  {z_noise.shape}")
print(f"Prior output z: {z_pred.shape}")   # (2, 512)

In [ ]:
# ── Cascaded Diffusion Decoder ──

class CascadedUpsampleBlock(nn.Module):
    """
    One stage of the cascaded upsampler.
    Takes low-res image + CLIP embedding, outputs higher-res image.
    """
    def __init__(self, in_ch=3, out_ch=3, base_ch=64, clip_dim=512):
        super().__init__()
        self.clip_proj = nn.Linear(clip_dim, base_ch * 4)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch + 1, base_ch, 3, padding=1),   # +1 for CLIP-cond channel
            nn.GroupNorm(8, base_ch), nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(base_ch, base_ch * 2, 3, padding=1),
            nn.GroupNorm(8, base_ch * 2), nn.SiLU(),
            nn.Conv2d(base_ch * 2, out_ch, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, x_low, z_clip):
        # Project CLIP embedding to spatial channel
        B, _, H, W = x_low.shape
        clip_ch = self.clip_proj(z_clip).mean(-1).unsqueeze(-1).unsqueeze(-1)
        clip_ch = clip_ch.expand(-1, 1, H, W)
        x_aug = torch.cat([x_low, clip_ch], dim=1)
        return self.net(x_aug)


# Demo cascade: 8×8 → 16×16 → 32×32
clip_emb = torch.randn(2, 512)

stage1 = CascadedUpsampleBlock(in_ch=4, out_ch=3, clip_dim=512)   # latent → 64px
stage2 = CascadedUpsampleBlock(in_ch=3, out_ch=3, clip_dim=512)   # 64 → 128px
stage3 = CascadedUpsampleBlock(in_ch=3, out_ch=3, clip_dim=512)   # 128 → 256px

x_latent = torch.randn(2, 4, 8, 8)
x_64  = stage1(x_latent, clip_emb)
x_128 = stage2(x_64,    clip_emb)
x_256 = stage3(x_128,   clip_emb)

print(f"Latent: {x_latent.shape} → 64px: {x_64.shape} → 128px: {x_128.shape} → 256px: {x_256.shape}")

In [ ]:
# ── Image Variation: the unique DALL-E 2 capability ──

# Because unCLIP conditions on CLIP *image* embeddings,
# you can vary any image by: encode → add noise → decode

def image_variation(encoder_fn, decoder_fn, image, num_variations=4, noise_level=0.1):
    """
    encoder_fn: maps image → CLIP image embedding
    decoder_fn: maps CLIP embedding → image
    noise_level: how much variation (0=identical, 1=very different)
    """
    z = encoder_fn(image)                              # (1, clip_dim)
    variations = []
    for _ in range(num_variations):
        z_noisy = z + noise_level * torch.randn_like(z)
        z_noisy = F.normalize(z_noisy, dim=-1)         # CLIP embeddings are L2-normalized
        variation = decoder_fn(z_noisy)
        variations.append(variation)
    return variations


# Visualize the concept (with mock encoder/decoder)
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 6, figsize=(14, 3))

# Mock original image
np.random.seed(42)
orig = np.random.rand(32, 32, 3) * 0.3 + 0.35
axes[0].imshow(orig)
axes[0].set_title("Original", fontsize=10)
axes[0].axis('off')

# Mock variations (with increasing noise)
noise_levels = [0.05, 0.15, 0.30, 0.50, 0.80]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 5))
for i, (ax, nl, c) in enumerate(zip(axes[1:], noise_levels, colors)):
    noisy = orig + nl * np.random.randn(*orig.shape)
    noisy = noisy.clip(0, 1)
    ax.imshow(noisy)
    ax.set_title(f"noise={nl}", fontsize=9)
    ax.axis('off')

plt.suptitle("DALL-E 2 Image Variations\n(same CLIP embedding + varying noise = different but semantically similar images)", fontsize=11)
plt.tight_layout(); plt.show()

## DALL-E 2 vs Stable Diffusion

| | DALL-E 2 (unCLIP) | Stable Diffusion (LDM) |
|--|-------------------|------------------------|
| **Conditioning** | CLIP image embedding (via prior) | CLIP text tokens (via cross-attention) |
| **Unique feature** | Image variations | Inpainting, img2img |
| **Latent space** | CLIP embedding space | VAE latent space |
| **Architecture** | Diffusion Transformer prior + Cascaded U-Net | U-Net with cross-attention |
| **Semantic control** | Embedding-level | Token-level |
| **Resolution** | 1024×1024 | 512×512 (v1), 768×768 (v2) |

## Summary

| Component | Description |
|-----------|-------------|
| **CLIP** | Shared image-text embedding space — the foundation |
| **Prior** | Diffusion Transformer: text z_t → image embedding z_i |
| **Decoder** | Cascaded diffusion: z_i → 64² → 256² → 1024² |
| **Image variation** | Encode any image to CLIP space → add noise → decode |
| **Semantic interpolation** | Lerp between two CLIP embeddings → morph images |

### Key Equations

**Prior training:**
$$L_{\text{prior}} = \mathbb{E}_{(x,y), t, \epsilon}\left[\|z_i - \hat{z}_i(z_i^{(t)}, t, z_t)\|^2\right]$$

**Decoder (unCLIP) objective:**
$$L_{\text{dec}} = \mathbb{E}_{x, t, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t, z_i, y)\|^2\right]$$

**CLIP image embedding (L2-normalized):**
$$z_i = E_{\text{image}}(x) / \|E_{\text{image}}(x)\|_2$$

## References

| Paper | Link |
|-------|------|
| unCLIP / DALL-E 2 | [arxiv 2204.06125](https://arxiv.org/abs/2204.06125) |
| CLIP — Learning Transferable Visual Models | [arxiv 2103.00020](https://arxiv.org/abs/2103.00020) |
| Cascaded Diffusion Models | [arxiv 2106.15282](https://arxiv.org/abs/2106.15282) |